# Process PA DMS data from Chen et al.

In [1]:
import os
import re
import numpy as np
import pandas as pd
from collections import defaultdict
from Bio import SeqIO
import subprocess

In [2]:
import yaml

_config_dir = os.path.dirname(os.path.abspath("../config.yaml"))
with open("../config.yaml") as _f:
    _config = yaml.safe_load(_f)
data_dir = os.path.normpath(os.path.join(_config_dir, _config["data_dir"]))

Parse the AA mutation column to extract wildtype AA, DMS site, and mutant AA. Exclude indel (`_`) and stop codon (`*`) mutations. Average the two no-drug fitness replicates (P1NO), group duplicate nucleotide mutations that produce the same AA change, and compute `dms_effect = log(mean fitness)`. Then align the DMS protein sequence (first 240 AA of PA) to the PA tree reference using MUSCLE to establish site numbering correspondence.

In [3]:
# Read reference PA protein from flu-usher
ref_fasta = os.path.join(data_dir, 'PA/all/curated_reference.fasta')
ref_seq = SeqIO.read(ref_fasta, 'fasta').seq
ref_aa_seq = str(ref_seq.translate()).replace('*', '')
print('Reference PA sequence length:', len(ref_aa_seq))

# Read DMS data from Excel
raw_dms = pd.read_excel('../data/dms_data/Chen_PA/fitness calculation.xlsx', sheet_name='HcMut')

# Parse AA mutation column: extract wt_aa, dms_site, mut_aa
# Exclude '_' (indel) and '*' (stop codon) mutations; keep only standard AA substitutions
valid_aa_pat = re.compile(r'^([A-Z])(\d+)([A-Z])$')
raw_dms['parsed'] = raw_dms['AA'].apply(lambda x: valid_aa_pat.match(str(x)))
raw_dms = raw_dms[raw_dms['parsed'].notna()].copy()
raw_dms['wt_aa']    = raw_dms['parsed'].apply(lambda m: m.group(1))
raw_dms['dms_site'] = raw_dms['parsed'].apply(lambda m: int(m.group(2)))
raw_dms['mut_aa']   = raw_dms['parsed'].apply(lambda m: m.group(3))

# Compute per-row mean no-drug fitness, then group by AA mutation and average
# (multiple nucleotide changes can produce the same AA change)
raw_dms['fitness'] = (raw_dms['P1NO-1-fit'] + raw_dms['P1NO-2-fit']) / 2.0
pa_dms_data = (
    raw_dms
    .groupby(['dms_site', 'wt_aa', 'mut_aa'], as_index=False)['fitness']
    .mean()
)
pa_dms_data['dms_effect'] = np.log(pa_dms_data['fitness'])

# Exclude synonymous AA mutations
pa_dms_data = pa_dms_data[pa_dms_data['wt_aa'] != pa_dms_data['mut_aa']].copy()

print('DMS site range:', pa_dms_data['dms_site'].min(), '-', pa_dms_data['dms_site'].max())

# Build DMS protein sequence from wt_aa at each site (sites 1-240)
dms_seq_df = (
    pa_dms_data
    .sort_values('dms_site')
    .drop_duplicates('dms_site')[['dms_site', 'wt_aa']]
)
dms_seq_df = dms_seq_df[dms_seq_df['dms_site'].between(1, 240)]

# Use reindex+dropna to handle missing sites; track the actual site numbers
# so the alignment numbering uses real site indices rather than a sequential counter
dms_site_series = dms_seq_df.set_index('dms_site')['wt_aa'].reindex(range(1, 241)).dropna()
aa_seq = ''.join(dms_site_series)
dms_site_list = dms_site_series.index.tolist()
print('DMS sequence length:', len(aa_seq))

# Save sequences for MUSCLE alignment
output_dir = '../results/dms_data/Chen_PA/'
os.makedirs(output_dir, exist_ok=True)
unaligned_fasta = os.path.join(output_dir, 'unaligned.fasta')
if not os.path.isfile(unaligned_fasta):
    with open(unaligned_fasta, 'w') as f:
        f.write('>reference\n' + ref_aa_seq + '\n')
        f.write('>dms\n'       + aa_seq       + '\n')

# Align the sequences using MUSCLE
aligned_fasta = os.path.join(output_dir, 'aligned.fasta')
if not os.path.isfile(aligned_fasta):
    cmd = ['muscle', '-align', unaligned_fasta, '-output', aligned_fasta]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)

Reference PA sequence length: 716


DMS site range: 1 - 240
DMS sequence length: 232


/home/hhaddox/miniforge3/envs/flu-mut-rates/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [4]:
# Read in the aligned sequences
seqs_dict = {}
aligned_records = list(SeqIO.parse(aligned_fasta, 'fasta'))
for record in aligned_records:
    seqs_dict[record.id] = str(record.seq)

aligned_ref_seq = seqs_dict['reference']
aligned_dms_seq = seqs_dict['dms']

# Compute percent identity
n_sites_to_compare = 0
n_identical = 0
for (ref_aa, dms_aa) in zip(aligned_ref_seq, aligned_dms_seq):
    if ref_aa != '-' and dms_aa != '-':
        n_sites_to_compare += 1
        if ref_aa == dms_aa:
            n_identical += 1

print(n_sites_to_compare, n_identical, n_identical / n_sites_to_compare)

# Determine numbering scheme.
# The DMS data may not cover all sites, so use the actual DMS site numbers
# (not a sequential counter) to ensure correct coordinate mapping.
numbering_dict = defaultdict(list)
assert len(aligned_ref_seq) == len(aligned_dms_seq)

# Reference: sequential counter starting at 1
ref_idx = 1
for (alignment_index, aa) in enumerate(aligned_ref_seq, 1):
    if aa != '-':
        numbering_dict['alignment_index'].append(alignment_index)
        numbering_dict['seq_id'].append('tree_reference_site')
        numbering_dict['seq_index'].append(ref_idx)
        numbering_dict['seq_aa'].append(aa)
        ref_idx += 1

# DMS: use actual DMS site numbers (not a sequential counter)
dms_char_idx = 0
for (alignment_index, aa) in enumerate(aligned_dms_seq, 1):
    if aa != '-':
        numbering_dict['alignment_index'].append(alignment_index)
        numbering_dict['seq_id'].append('dms_site')
        numbering_dict['seq_index'].append(dms_site_list[dms_char_idx])
        numbering_dict['seq_aa'].append(aa)
        dms_char_idx += 1

alignment_numbering_df = (
    pd.DataFrame(numbering_dict)
    .pivot(index='alignment_index', columns='seq_id', values='seq_index')
    .reset_index()
    .rename_axis(None, axis=1)
    .dropna()
    [['dms_site', 'tree_reference_site']]
)
alignment_numbering_df['dms_site'] = alignment_numbering_df['dms_site'].astype(int)
alignment_numbering_df['tree_reference_site'] = alignment_numbering_df['tree_reference_site'].astype(int)
alignment_numbering_df.head()

232 217 0.9353448275862069


,dms_site,tree_reference_site
0,1,1
1,2,2
3,4,4
4,5,5
5,6,6


In [5]:
for i, (ref_aa, dms_aa) in enumerate(zip(aligned_ref_seq, aligned_dms_seq)):
    if i > 241:
        continue
    if ref_aa != dms_aa:
        print(i, ref_aa, dms_aa, '*')
    else:
        print(i, ref_aa, dms_aa)

0 M M
1 E E
2 D - *
3 F F
4 V V
5 R R
6 Q Q
7 C C
8 F F
9 N N
10 P P
11 M M
12 I I
13 V V
14 E E
15 L L
16 A A
17 E E
18 K K
19 A A
20 M M
21 K K
22 E E
23 Y Y
24 G G
25 E E
26 D D
27 L P *
28 K K
29 I I
30 E E
31 T T
32 N N
33 K K
34 F F
35 A A
36 A A
37 I I
38 C C
39 T T
40 H H
41 L L
42 E E
43 V V
44 C C
45 F F
46 M M
47 Y Y
48 S S
49 D D
50 F F
51 H H
52 F F
53 I I
54 N D *
55 E E
56 Q R *
57 G G
58 E E
59 S S
60 I I
61 V I *
62 V V
63 E E
64 L S *
65 D G *
66 D D
67 P P
68 N N
69 A A
70 L L
71 L L
72 K K
73 H H
74 R R
75 F F
76 E E
77 I I
78 I I
79 E E
80 G - *
81 R - *
82 D - *
83 R - *
84 T I *
85 M M
86 A A
87 W W
88 T T
89 V V
90 V V
91 N N
92 S S
93 I I
94 C C
95 N N
96 T T
97 T T
98 G G
99 A V *
100 E E
101 K K
102 P P
103 K K
104 F F
105 L L
106 P P
107 D D
108 L L
109 Y Y
110 D D
111 Y Y
112 K K
113 E E
114 N N
115 R R
116 F F
117 I I
118 E E
119 I I
120 G G
121 V V
122 T T
123 R R
124 R R
125 E E
126 V V
127 H H
128 I I
129 Y Y
130 Y Y
131 L L
132 E E
133 K K
134 A A
135 

In [6]:
# Merge DMS data with numbering scheme and save
pa_dms_data_processed = (
    pa_dms_data
    .merge(alignment_numbering_df, on='dms_site', validate='many_to_one')
)

pa_dms_data_processed = pa_dms_data_processed[['dms_site', 'wt_aa', 'mut_aa', 'tree_reference_site', 'dms_effect']]
pa_dms_data_processed.to_csv(os.path.join(output_dir, 'processed_dms_data.csv'), index=False)
print('Number of mutations with processed data:', len(pa_dms_data_processed))
pa_dms_data_processed.head()

Number of mutations with processed data: 666


,dms_site,wt_aa,mut_aa,tree_reference_site,dms_effect
0,1,M,I,1,-2.235865
1,1,M,L,1,-3.317741
2,2,E,D,2,-1.132207
3,4,F,L,4,-0.058706
4,4,F,S,4,-2.619114


In [7]:
pa_dms_data_processed.tail()

,dms_site,wt_aa,mut_aa,tree_reference_site,dms_effect
661,239,N,D,239,-2.026654
662,239,N,I,239,0.028892
663,239,N,S,239,-1.403523
664,239,N,Y,239,-0.003344
665,240,G,S,240,-1.759862


In [8]:
sum(pa_dms_data_processed['dms_site'] != pa_dms_data_processed['tree_reference_site'])

0